# ضبط نموذج Gemma على اللهجة العراقية (Fine-Tuning) — النسخة المصحّحة

هذا الدفتر يدرّب نموذج `google/gemma-4-E4B-it` على بيانات اللهجة العراقية باستخدام تقنية **LoRA** ومكتبة **TRL (SFTTrainer)**.

## 🔧 التصحيحات في هذه النسخة (مقارنة بالنسخة السابقة التي أنتجت مخرجات عشوائية)

1. **حذف `model.config.use_cache = False`** من خلية التدريب — هذا السطر يفسد حسابات الانتباه في Gemma 4 E4B لأن معماريتها تشارك قيم K/V بين الطبقات عبر الـ cache (transformers issue #45242). هذا كان السبب الجذري للمخرجات العشوائية.
2. **تحديث transformers لآخر إصدار** في خلية التثبيت (الإصلاح الرسمي للباگ).
3. **استبدال `target_modules="all-linear"` بقائمة صريحة** (q/k/v/o/gate/up/down) — أسلم لمعمارية E4B.
4. **إزالة الاستيرادات المكررة** في خلية LoRA.
5. **إضافة نقطة فحص للـ loss** — إذا كان عالقاً عند 10–15 فالمشكلة ما زالت قائمة.

البيانات تجمع مصدرين من مستودع `fine_tuning/`:
- قاموس مصطلحات (`word.json`): كلمة↔معنى وأسئلة حسب الفئة.
- عينة متوازنة من محادثات بيع/شراء وحياة يومية حقيقية متعددة الأدوار (20 فئة) من مستودع `database_LLm`.

المجموع: **37,259** مثال تدريب (`train_chat.jsonl`).

**خطوات العمل:** تثبيت المكتبات ← إعادة تشغيل الجلسة ← تحميل البيانات وتجهيزها ← تحميل النموذج الأساسي واختباره قبل التدريب ← تحليل أطوال البيانات ← إعدادات التدريب ← التدريب ← رسم منحنى الخسارة ← تحميل النموذج المدرّب واختباره.


## 1. تثبيت المكتبات المطلوبة

تثبيت PyTorch ومكتبات Hugging Face: `transformers` للنماذج، `datasets` للبيانات، `trl` للتدريب بأسلوب SFT، و`peft` لتقنية LoRA، إضافة إلى `torchao` (مطلوبة لطبقات Gemma المخصصة) و`matplotlib` لرسم منحنى الخسارة لاحقاً.

In [1]:
# تثبيت PyTorch وأداة TensorBoard لمتابعة التدريب
%pip install torch tensorboard

# تثبيت مكتبات Hugging Face اللازمة للتدريب
%pip install datasets accelerate evaluate trl peft protobuf sentencepiece

# ⚠️ مهم جداً: تحديث transformers لآخر إصدار
# الإصدارات القديمة فيها باگ يفسد حسابات الانتباه في Gemma 4 E2B/E4B
# عند use_cache=False (راجع issue #45242 في transformers)
%pip install -U transformers

# تحديث torchao (مطلوبة لطبقات Gemma المخصصة) وتثبيت matplotlib للرسوم البيانية
%pip install --upgrade torchao matplotlib

# 🔄 بعد التثبيت: أعد تشغيل الجلسة (Runtime > Restart session) ثم كمّل من الخلية التالية


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 72.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 219.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

## 2. تحميل مستودع البيانات

استنساخ مستودع GitHub الذي يحتوي على ملفات بيانات التدريب باللهجة العراقية بصيغة JSONL.

In [1]:
!git clone https://github.com/ameer20042005/database_LLm.git

# Now you can access the files in the 'database_LLm' directory
# For example, to list the contents:
# !ls database_LLm

Cloning into 'database_LLm'...
remote: Enumerating objects: 1977, done.
remote: Counting objects: 100% (72/72), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 1977 (delta 38), reused 50 (delta 23), pack-reused 1905 (from 2)
Receiving objects: 100% (1977/1977), 60.69 MiB | 27.86 MiB/s, done.
Resolving deltas: 100% (306/306), done.


## 3. قراءة البيانات وتجهيزها

قراءة مصدرين من `data/`:

- **v4** — `iraqi_train_v4_part01.jsonl` … `part03.jsonl` (163,429 محادثة) و`iraqi_val_v4.jsonl` (10,330 محادثة): محادثات اللهجة العراقية العامة (بيع/شراء، يومي، اجتماعي)، مُراجَعة (تسريبات الفصحى مصلَّحة، تغطية مفردات `word.json` 84%).
- **v5 (جديد)** — `iraqi_train_v5_part01.jsonl` … `part03.jsonl` (19,000 محادثة) و`iraqi_val_v5.jsonl` (1,000 محادثة): بيانات **كتالوج مؤرضن (grounded)** — كل مثال فيه رسالة `system` تحتوي قائمة أسعار/منتجات عشوائية، وجواب `assistant` مُلزَم يقتبس الاسم والسعر حرفياً منها (أو يرفض بأدب إذا المنتج مو موجود). الهدف: تدريب الموديل يقرأ الأرقام من الـ context بدل ما يحفظها كـ mapping ثابت (منتج → سعر)، لتقليل **انجراف الأرقام** (Number Drift) وقت التوليد. راجع `generate_v5_grounded_data.py` و`data/product_bank_v5.json` بمستودع `database_LLm` للتفاصيل.

المجموع بعد الدمج: **182,429** مثال تدريب و**11,330** مثال اختبار.

`data/greetings_smalltalk_only.jsonl` **غير مُحمَّل هنا عمداً**: هو نسخة مطابقة 100% (نفس المعرّفات ونفس المحتوى) من محادثات التحية الموجودة أصلاً داخل ملفات train/val (v4 وv5)، ومُبقى فقط للمراجعة اليدوية — تحميله هنا يضاعف وزن فئة التحية أثناء التدريب.

التأكد من وجود عمود `messages` (وإن لم يوجد يُبنى من عمودي Question/Answer)، ثم تحويل البيانات إلى صيغة HuggingFace Dataset. ملاحظة: تقسيم train/test هنا جاهز من ملفات v4/v5 نفسها (لا تقسيم 80/20 إضافي بالكود).

In [5]:
import os
import glob
import pandas as pd
from datasets import Dataset, DatasetDict

# مسارات ملفات التدريب (data/) — v4 (اللهجة العراقية العامة) + v5 (كتالوج مؤرضن لمنع انجراف الأسعار)
# مقسّمة لأجزاء (parts) لأن أي ملف يتجاوز 100 ميجا يُرفض عند git push
train_files = (
    sorted(glob.glob("/content/database_LLm/data/iraqi_train_v4_part*.jsonl")) +
    sorted(glob.glob("/content/database_LLm/data/iraqi_train_v5_part*.jsonl"))
)

# مسارات ملفات الاختبار (v4 + v5)
val_files = [
    "/content/database_LLm/data/iraqi_val_v4.jsonl",
    "/content/database_LLm/data/iraqi_val_v5.jsonl",
]

# ملاحظة: data/greetings_smalltalk_only.jsonl مستبعد عمداً هنا لأنه نسخة
# مطابقة 100% من محادثات التحية الموجودة أصلاً داخل ملفات train/val أعلاه
# (مخصص للمراجعة اليدوية فقط) — ضمّه هنا يضاعف وزن فئة التحية أثناء التدريب.

print(f"عدد أجزاء ملف التدريب الموجودة: {len(train_files)}")
print("جاري قراءة ملفات التدريب والاختبار...")

try:
    # قراءة وتجميع بيانات التدريب (v4 + v5)
    train_dfs = [pd.read_json(f, lines=True) for f in train_files]
    train_df = pd.concat(train_dfs, ignore_index=True)

    # قراءة وتجميع بيانات الاختبار (v4 + v5)
    val_dfs = [pd.read_json(f, lines=True) for f in val_files]
    val_df = pd.concat(val_dfs, ignore_index=True)

    # عدد أمثلة v5 (نظام الكتالوج المؤرضن) ضمن كل تقسيم — للتأكد من دخولها فعلياً
    v5_train_n = train_df["source_file"].astype(str).str.contains("generate_v5").sum() if "source_file" in train_df.columns else 0
    v5_val_n = val_df["source_file"].astype(str).str.contains("generate_v5").sum() if "source_file" in val_df.columns else 0
    print(f"منها من v5 (كتالوج مؤرضن): {v5_train_n} تدريب / {v5_val_n} اختبار")

    print("تم قراءة الملفات بنجاح.")
except Exception as e:
    raise ValueError(f"Error reading files: {e}")

# دالة مساعدة للتأكد من عمود الرسائل
def process_df(df):
    if 'messages' not in df.columns:
        if 'Question' in df.columns and 'Answer' in df.columns:
            df['messages'] = df.apply(lambda row: [
                {"role": "user", "content": str(row['Question'])},
                {"role": "assistant", "content": str(row['Answer'])}
            ], axis=1)
        else:
            raise ValueError("الملف لا يحتوي على عمود 'messages' ولا يحتوي على 'Question' و 'Answer' لتحويلها.")
    return df

# معالجة البيانات
train_df = process_df(train_df)
val_df = process_df(val_df)

# تحويل البيانات إلى صيغة HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# إزالة الأعمدة الإضافية للحفاظ على التناسق
train_cols_to_remove = [col for col in train_dataset.column_names if col != 'messages']
val_cols_to_remove = [col for col in val_dataset.column_names if col != 'messages']

train_dataset = train_dataset.remove_columns(train_cols_to_remove)
val_dataset = val_dataset.remove_columns(val_cols_to_remove)

# تجميع البيانات في DatasetDict
dataset = DatasetDict({
    'train': train_dataset,
    'test': val_dataset
})

# طباعة شكل البيانات النهائي وإحصائياتها
print(f"\nإجمالي عينات التدريب: {len(dataset['train'])}")
print(f"إجمالي عينات الاختبار: {len(dataset['test'])}")
print("\n--- مثال على شكل المحادثة المجهزة للتدريب ---")
print(dataset["train"][0]["messages"])

عدد أجزاء ملف التدريب الموجودة: 6
جاري قراءة ملفات التدريب والاختبار...
منها من v5 (كتالوج مؤرضن): 19950 تدريب / 1050 اختبار
تم قراءة الملفات بنجاح.

إجمالي عينات التدريب: 183379
إجمالي عينات الاختبار: 11380

--- مثال على شكل المحادثة المجهزة للتدريب ---
[{'role': 'user', 'content': 'هلا، شنو عندكم من طاولة سفرة؟'}, {'role': 'assistant', 'content': 'عندنا طاولة سفرة ممتاز. زجاج وستيل، 6 كراسي. بـ576,000 دينار'}, {'role': 'user', 'content': 'ما تنزل علي؟'}, {'role': 'assistant', 'content': 'والله على خاطرك، هذا آخر سعر'}, {'role': 'user', 'content': 'ماشي'}, {'role': 'assistant', 'content': 'إن شاء الله'}]


In [7]:
import pprint

pprint.pp(dataset["train"][0]["messages"])

[{'role': 'user', 'content': 'هلا، شنو عندكم من طاولة سفرة؟'},
 {'role': 'assistant',
  'content': 'عندنا طاولة سفرة ممتاز. زجاج وستيل، 6 كراسي. بـ576,000 دينار'},
 {'role': 'user', 'content': 'ما تنزل علي؟'},
 {'role': 'assistant', 'content': 'والله على خاطرك، هذا آخر سعر'},
 {'role': 'user', 'content': 'ماشي'},
 {'role': 'assistant', 'content': 'إن شاء الله'}]


## 4. تسجيل الدخول وتحميل النموذج الأساسي

تسجيل الدخول إلى Hugging Face عبر التوكن المخزّن في أسرار Colab (المفتاح `HF_TOKEN`)، ثم تحميل نموذج `google/gemma-4-E4B-it` مع مقطّع الكلمات (Tokenizer) الخاص به.

> **ملاحظة:** يجب الموافقة على شروط استخدام Gemma في حسابك على Hugging Face قبل التحميل.

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from google.colab import userdata

# 1. تسجيل الدخول إلى Hugging Face
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("تم تسجيل الدخول إلى Hugging Face بنجاح.")
except Exception as e:
    print("⚠️ تنبيه هام: لم يتم العثور على 'HF_TOKEN'. يرجى إضافته في قسم Secrets 🔑 للموافقة على شروط نموذج Gemma.")

# 2. تعريف اسم النموذج لتفادي خطأ NameError
base_model = "google/gemma-4-E4B-it"

# 3. تحميل النموذج ومقطّع الكلمات (Tokenizer)
print(f"\nجاري تحميل النموذج {base_model}...")
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype="auto",
    device_map="auto",
    attn_implementation="eager"
)
tokenizer = AutoTokenizer.from_pretrained(base_model)

print(f"\nDevice: {model.device}")
print(f"DType: {model.dtype}")

تم تسجيل الدخول إلى Hugging Face بنجاح.

جاري تحميل النموذج google/gemma-4-E4B-it...


config.json:   0%|          | 0.00/5.14k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/17.3k [00:00<?, ?B/s]


Device: cuda:0
DType: torch.bfloat16


## 5. اختبار النموذج الأساسي قبل التدريب

تجربة النموذج على عينة عشوائية من بيانات الاختبار ومقارنة إجابته بالإجابة الأصلية، لمعرفة مستوى النموذج **قبل** الضبط. لاحظ أن إجاباته طويلة وعامة ولا تلتزم بأسلوب البيانات المختصر.

In [9]:
from transformers import pipeline

from random import randint
import re

# Load the model and tokenizer into the pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Load a random sample from the test dataset
rand_idx = randint(0, len(dataset["test"])-1)
test_sample = dataset["test"][rand_idx]

# Convert as test example into a prompt with the Gemma template (exclude the assistant response)
prompt = pipe.tokenizer.apply_chat_template(test_sample["messages"][:-1], tokenize=False, add_generation_prompt=True)
outputs = pipe(prompt, max_new_tokens=256, disable_compile=True)

# Extract the last user query and original answer (works for single-turn glossary
# examples and multi-turn sales/social conversations alike)
print(f"Question:\n{test_sample['messages'][-2]['content']}\n")
print(f"Original Answer:\n{test_sample['messages'][-1]['content']}\n")
print(f"Generated Answer (base model):\n{outputs[0]['generated_text'][len(prompt):].strip()}")

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Question:
الرسوم؟

Original Answer:
23,000 دينار والمدة 10 أيام

Generated Answer (base model):
**تعتمد الرسوم على الجهة التي ستقوم بتغيير العنوان (مثل الأحوال المدنية، أو منصة حكومية معينة)، وكذلك على قوانين بلدك.**

**لذلك، لكي أتمكن من إعطائك إجابة دقيقة، هل يمكنك إخباري:**

1. **في أي بلد أنت؟** (أو أي دولة تقيم فيها)
2. **ما هي الجهة التي تود التقديم لديها لتغيير العنوان؟** (مثلاً: الأحوال المدنية، أو وزارة الداخلية، أو منصة حكومية معينة)

بمجرد معرفة هذه التفاصيل، يمكنني إرشادك إلى الرسوم المتوقعة أو الإجراءات المطلوبة.<turn|>


## 6. تجربة محادثة حرة مع النموذج الأساسي

اختبار سريع بسؤال عام للتأكد من عمل خط الأنابيب (Pipeline) بشكل صحيح.

In [10]:
outputs = pipe([{"role": "user", "content": "مرحبا، عندي استفسار بخصوص شغلة."}], max_new_tokens=256, disable_compile=True)
print(outputs[0]['generated_text'][-1]['content'])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


أهلاً بك! تفضل/تفضلي، أنا هنا للمساعدة. ما هو استفسارك؟ 😊


## 7. تجربة التوجيه بالأمثلة (Few-shot Prompting)

تزويد النموذج بمثالين فقط من بيانات الاختبار كتوجيه ثم طرح سؤال جديد، لتوضيح الفرق بين التوجيه بالأمثلة والضبط الكامل (Fine-tuning).

In [11]:
message = []

# few shot prompt: أخذ مثالين فقط لتوجيه النموذج بدلاً من كامل مجموعة البيانات لتفادي انهيار الذاكرة
for i in range(2):
  item = dataset['test'][i]
  message.append(
      {"role": "user", "content": item["messages"][1]["content"]}
  )
  message.append(
      {"role": "assistant", "content": item["messages"][2]["content"]}
  )

# actual question (سؤال لتجربة النموذج)
message.append(
    {"role": "user", "content": "أريد أشتري سيارة اقتصادية ومناسبة لعائلتي، شنو تنصحني؟"}
)

outputs = pipe(message, max_new_tokens=256, disable_compile=True)
# طباعة الرد الأخير فقط
print(outputs[0]['generated_text'][-1]['content'])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


بما أنك تبحث عن سيارة **اقتصادية ومناسبة لعائلة**، فهناك عدة عوامل يجب أن نأخذها في الحسبان لتحديد الخيار الأفضل لك، مثل:

1. **حجم العائلة:** كم عدد الأفراد الذين سيسافرون بانتظام؟
2. **طبيعة الاستخدام:** هل القيادة في المدينة بشكل أساسي، أم تتطلب سفر لمسافات طويلة (طرق وعرة، طرق سريعة)؟
3. **الميزانية:** ما هو النطاق السعري التقريبي الذي تفكر فيه؟
4. **الأولوية:** هل الأولوية القصوى هي **أقل استهلاك للوقود**، أم **مساحة داخلية أكبر**، أم **المتانة**؟

لكن بشكل عام، بناءً على المعايير التي ذكرتها (اقتصادية وعائلية)، إليك بعض **الفئات والخيارات المقترحة** التي تحظى بشعبية في هذا المجال:

---

### 🚗 الخيارات الأكثر اقتصادية (للمدن والاستخدام اليومي)

إذا كانت أولويتك هي **توفير الوقود** والاستخدام داخل المدينة


## 8. تحليل أطوال التسلسلات بعد التقطيع

حساب عدد الرموز (Tokens) لكل عينة بعد تطبيق قالب محادثة Gemma، واستخراج إحصائيات (الحد الأدنى/الأقصى، المتوسط، النسبتان المئويتان 95 و99) لاختيار قيمة `max_length` مناسبة تغطي أغلب البيانات دون هدر للذاكرة.

الناتج يُخزَّن في المتغير `optimized_max_length` ويُستخدم في خلية إعدادات التدريب التالية.

In [12]:
from tqdm import tqdm
import numpy as np

# حساب طول كل عينة بعد التقطيع (Tokenization) باستخدام قالب المحادثة الخاص بـ Gemma
tokenized_lengths = []
for split in ['train', 'test']:
    for example in tqdm(dataset[split], desc=f"Calculating token lengths for {split} split"):
        messages = []
        for msg in example['messages']:
            # قالب Gemma يستخدم الدور "model" بدلاً من "assistant"
            role = "model" if msg["role"] == "assistant" else msg["role"]
            messages.append({"role": role, "content": msg["content"]})
        text_formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        tokenized_output = tokenizer(text_formatted, add_special_tokens=True)
        tokenized_lengths.append(len(tokenized_output['input_ids']))

# حساب الإحصائيات
min_len = np.min(tokenized_lengths)
max_len = np.max(tokenized_lengths)
avg_len = np.mean(tokenized_lengths)
p95_len = np.percentile(tokenized_lengths, 95)
p99_len = np.percentile(tokenized_lengths, 99)

print(f"\nMin token length: {min_len}")
print(f"Max token length: {max_len}")
print(f"Average token length: {avg_len:.2f}")
print(f"95th percentile token length: {p95_len:.2f}")
print(f"99th percentile token length: {p99_len:.2f}")

# اختيار max_length مناسب: 512 كقيمة افتراضية، وإذا تجاوزتها نسبة الـ 99%
# نرفعها إلى أقرب قوة للعدد 2 (بحد أقصى 2048 حفاظاً على الذاكرة)
optimized_max_length = 512
if p99_len > 512:
    optimized_max_length = min(2 ** int(np.ceil(np.log2(p99_len))), 2048)

print(f"\nSuggested `max_length` for SFTConfig: {optimized_max_length}")

Calculating token lengths for test split: 100%|██████████| 11380/11380 [00:04<00:00, 2658.48it/s]


Min token length: 21
Max token length: 454
Average token length: 127.58
95th percentile token length: 285.00
99th percentile token length: 356.00

Suggested `max_length` for SFTConfig: 512


## 9. إعدادات التدريب (SFTConfig)

تحديد معاملات التدريب: 5 حقب (Epochs)، دفعة بحجم 8، معدل تعلم `1e-4` مع مجدول `cosine` وفترة إحماء 10%، وحفظ وتقييم في نهاية كل حقبة مع الاحتفاظ بأفضل نموذج حسب `eval_loss`.

> ⚠️ **تنبيه خاص بـ Gemma 4 E4B:** يجب أن يبقى `gradient_checkpointing=False`، لأن تفعيله يفرض `use_cache=False` داخلياً، وهذا يفسد حسابات الانتباه في معمارية E4B (مشاركة KV بين الطبقات) وينتج نموذجاً مخرجاته عشوائية.

قيمة `max_length` تؤخذ من المتغير `optimized_max_length` المحسوب في خلية التحليل السابقة.


In [13]:
from trl import SFTConfig
import torch

checkpoint_dir = "./gemma-iraqi-finetune"  # مجلد حفظ نقاط التدريب (Checkpoints)
learning_rate = 5e-5

torch_dtype = model.dtype

args = SFTConfig(
    output_dir=checkpoint_dir,
    max_length=optimized_max_length,  # الطول المقترح من خلية تحليل الأطوال
    packing=False,
    num_train_epochs=1,
    per_device_train_batch_size=8,  # A100 40GB تكفي
    gradient_accumulation_steps=1,
    # ⚠️ لا تفعّل gradient_checkpointing مع Gemma 4 E4B:
    # تفعيله يفرض use_cache=False ويفسد حسابات الانتباه (issue #45242)
    gradient_checkpointing=False,
    optim="adamw_torch_fused",
    logging_steps=1,
    save_strategy="epoch",
    eval_strategy="epoch",
    # حفظ أفضل نموذج حسب خسارة التقييم
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=learning_rate,
    fp16=True if torch_dtype == torch.float16 else False,
    bf16=True if torch_dtype == torch.bfloat16 else False,
    # مجدول cosine مع فترة إحماء لاستقرار التدرجات
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    push_to_hub=False,
    report_to="tensorboard",
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": True,
    }
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## 10. تجهيز LoRA وإنشاء المدرب (SFTTrainer)

ضبط رمز الحشو (Padding) للـ Tokenizer، وتعريف إعدادات LoRA، ثم تجهيز عمود `text` بتطبيق قالب محادثة Gemma على البيانات، وأخيراً إنشاء `SFTTrainer` وطباعة عدد المعاملات القابلة للتدريب.

> ⚠️ **تعديل مهم:** استُبدل `target_modules="all-linear"` بقائمة صريحة لطبقات الانتباه والـ MLP فقط (`q/k/v/o_proj` و`gate/up/down_proj`). خيار `all-linear` في E4B يلمس طبقات معمارية حساسة (AltUp، الإسقاطات لكل طبقة، وأبراج الرؤية/الصوت) ويؤدي لنتائج غير مستقرة. القائمة الصريحة هي الموصى بها رسمياً لـ Gemma 4.


In [14]:
from trl import SFTTrainer
from peft import LoraConfig

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # regex يطابق طبقات نموذج اللغة فقط — يتجاهل برجي الرؤية والصوت
    target_modules=r".*language_model.*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
)

print("Filtering empty or invalid rows...")
dataset = dataset.filter(lambda x: x["messages"] is not None and len(x["messages"]) > 0)

def apply_template(split):
    formatted_texts = []
    for example in dataset[split]:
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
        formatted_texts.append(text)
    return formatted_texts

print("Applying chat template to dataset...")
for split in dataset.keys():
    texts = apply_template(split)
    if "text" in dataset[split].column_names:
        dataset[split] = dataset[split].remove_columns("text")
    dataset[split] = dataset[split].add_column("text", texts)

args.dataset_text_field = "text"

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    peft_config=peft_config,
)

print("\n--- Model Parameters Info ---")
trainer.model.print_trainable_parameters()


Filtering empty or invalid rows...


Filter:   0%|          | 0/183379 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11380 [00:00<?, ? examples/s]

Applying chat template to dataset...


Flattening the indices:   0%|          | 0/183379 [00:00<?, ? examples/s]

...

Truncating train dataset:   0%|          | 0/183379 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/11380 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/11380 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/11380 [00:00<?, ? examples/s]


--- Model Parameters Info ---
trainable params: 34,881,536 || all params: 7,975,982,368 || trainable%: 0.4373


## 11. بدء التدريب

تنظيف ذاكرة الـ GPU ثم تشغيل التدريب وحفظ النموذج النهائي عند الانتهاء.

> 🛑 **أهم تعديل في هذا الدفتر:** حُذف السطر `model.config.use_cache = False` نهائياً.
>
> في Gemma 4 E4B، آخر 18 طبقة لا تحسب K وV بنفسها بل تستعيرها من الطبقات الأولى **عبر الـ KV cache**. تعطيل الـ cache يقطع قناة النقل هذه، فتُحسب قيم بديلة خاطئة وتصبح الـ logits فاسدة طوال التدريب — والنتيجة adapter يولّد كلاماً عشوائياً. (transformers issue #45242)
>
> ✅ **نقطة فحص مبكرة:** راقب الـ loss في أول 100–200 خطوة. القيمة الطبيعية لـ E4B تبدأ حوالي 2–5 وتنخفض. إذا شفتها عالقة عند 10–15 أو أعلى، أوقف التدريب فوراً — معناها المشكلة ما زالت موجودة (غالباً إصدار transformers قديم).


In [ ]:
import torch
import gc

# تنظيف ذاكرة الـ GPU والذاكرة العشوائية قبل بدء التدريب
torch.cuda.empty_cache()
gc.collect()

# 🛑 لا تضف model.config.use_cache = False هنا!
# هذا السطر (الموجود بأغلب دروس LoRA) يفسد Gemma 4 E4B تحديداً
# لأن معماريتها تعتمد على الـ KV cache لمشاركة قيم K/V بين الطبقات.
# نتأكد أنه مفعّل صراحة:
model.config.use_cache = True

# تفعيل تدرجات المدخلات لحل مشكلة التدرجات المفقودة مع PEFT
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# معلومات سريعة قبل البدء
print(f"\nTrain dataset size: {len(dataset['train'])}")
print(f"Eval dataset size: {len(dataset['test'])}")
print(f"Learning rate configured: {trainer.args.learning_rate}")
print(f"use_cache = {model.config.use_cache}  (يجب أن يكون True)")
print("\nStarting training...")

# بدء التدريب
train_result = trainer.train()

# طباعة إحصائيات التدريب
print("\nTraining complete!")
print(train_result.metrics)

# حفظ النموذج النهائي (أفضل نقطة حسب eval_loss بفضل load_best_model_at_end)
trainer.save_model()


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.



Train dataset size: 183379
Eval dataset size: 11380
Learning rate configured: 5e-05
use_cache = True  (يجب أن يكون True)

Starting training...


Epoch,Training Loss,Validation Loss


## 12. رسم منحنى الخسارة

رسم خسارة التدريب والتقييم لكل حقبة من سجل المدرب، للتأكد من أن النموذج يتعلم دون إفراط في التخصيص (Overfitting).

> **تنبيه:** يجب تشغيل خلية التدريب (الخلية 11) أولاً، وإلا سيظهر خطأ `trainer` غير معرّف.

In [ ]:
import matplotlib.pyplot as plt

# Access the log history
log_history = trainer.state.log_history

# Extract training / validation loss
train_losses = [log["loss"] for log in log_history if "loss" in log]
epoch_train = [log["epoch"] for log in log_history if "loss" in log]
eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]
epoch_eval = [log["epoch"] for log in log_history if "eval_loss" in log]

# Plot the training loss
plt.plot(epoch_train, train_losses, label="Training Loss")
plt.plot(epoch_eval, eval_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()


## 13. تحميل النموذج المدرّب من آخر نقطة حفظ

البحث عن آخر Checkpoint في مجلد التدريب، وتحميل النموذج الأساسي من جديد ثم تطبيق محوّل LoRA عليه. مفيد أيضاً لاستئناف العمل بعد إعادة تشغيل جلسة Colab دون إعادة التدريب.

In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

checkpoint_dir = "./gemma-iraqi-finetune"
base_model_id = "google/gemma-4-E4B-it"

# 1. البحث عن آخر نقطة حفظ (Checkpoint)
if not os.path.exists(checkpoint_dir):
    print("Checkpoint directory not found. Please run training first.")
else:
    subdirs = [os.path.join(checkpoint_dir, d) for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint")]
    if not subdirs:
        print("No checkpoints found! You need to let the training run long enough to save at least one checkpoint.")
    else:
        latest_checkpoint = max(subdirs, key=os.path.getmtime)
        print(f"Loading adapter from: {latest_checkpoint}")

        # 2. تحميل النموذج الأساسي
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            torch_dtype="auto",
            device_map="auto",
            attn_implementation="eager"
        )
        tokenizer = AutoTokenizer.from_pretrained(base_model_id)

        # 3. تحميل محوّل LoRA وتطبيقه على النموذج الأساسي
        model = PeftModel.from_pretrained(base_model, latest_checkpoint)
        print("PEFT Model loaded successfully!")

## 14. اختبار النموذج بعد الضبط

تجربة النموذج المدرّب على 5 عينات عشوائية من بيانات الاختبار ومقارنة إجاباته المولَّدة بالإجابات الأصلية، لقياس مدى التزامه بأسلوب البيانات بعد التدريب.

In [ ]:
from transformers import pipeline
import random

# Load the model and tokenizer into the pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

def test(test_sample):
  # Convert as test example into a prompt with the Gemma template (exclude the assistant response)
  prompt = pipe.tokenizer.apply_chat_template(test_sample["messages"][:-1], tokenize=False, add_generation_prompt=True)
  outputs = pipe(prompt, max_new_tokens=256, disable_compile=True)

  # Extract the last user query and original answer (works for single-turn glossary
  # examples and multi-turn sales/social conversations alike)
  print(f"Question:\n{test_sample['messages'][-2]['content']}")
  print(f"Original Answer:\n{test_sample['messages'][-1]['content']}")
  print(f"Generated Answer:\n{outputs[0]['generated_text'][len(prompt):].strip()}")
  print("-"*80)

# Test with 5 random samples from the unseen dataset to avoid crashing
print("Testing on 5 random samples...\n")
for _ in range(5):
  rand_idx = random.randint(0, len(dataset['test']) - 1)
  test(dataset['test'][rand_idx])

## 15. تجربة محادثة حرة مع النموذج المدرّب

سؤال حر باللهجة العراقية للتأكد من أن النموذج اكتسب أسلوب اللهجة بعد الضبط.

In [ ]:
IRAQI_SYSTEM_PROMPT = """
أنت مساعد ذكاء اصطناعي عراقي.

القواعد:
- تكلم دائماً باللهجة العراقية الطبيعية.
- استخدم مفردات عراقية مثل: شنو، شلون، هسه، كلش، مو، أني، إنت، خوش، بعد، عبالك.
- لا تستخدم العربية الفصحى إلا إذا طلب المستخدم ذلك.
- إذا كتب المستخدم بالفصحى، أجب بالعراقي أيضاً ما لم يطلب غير ذلك.
- حافظ على الأسلوب الودود والمحترم والمساعد.
- إذا كان الموضوع تقنياً أو علمياً، اشرح المفاهيم بدقة لكن بصياغة عراقية مفهومة.
"""

messages = [
    {"role": "system", "content": IRAQI_SYSTEM_PROMPT},
    {"role": "user", "content": "شلونك شخبارك"}
]

outputs = pipe(
    messages,
    max_new_tokens=256,
    disable_compile=True,
)

print(outputs[0]["generated_text"][-1]["content"])

In [ ]:
import os
from huggingface_hub import HfApi
from google.colab import userdata

# 1. جلب التوكن من الأسرار بدلاً من كتابته يدوياً لتجنب أخطاء النصوص
try:
    hf_token = userdata.get('ameerwisam2005')
    if not hf_token or not hf_token.startswith('hf_'):
        raise ValueError("التوكن غير صالح. تأكد أن السر 'ameerwisam2005' يحتوي على توكن يبدأ بـ hf_")
except Exception as e:
    print("⚠️ لم يتم العثور على السر 'ameerwisam2005' أو أنه غير صالح. تأكد من تفعيل Notebook access.")
    raise e

# 2. مسار حفظ النموذج محلياً
local_save_path = "./final_model_to_upload"

print("جاري حفظ النموذج ومقطع الكلمات (Tokenizer) محلياً...")
model.save_pretrained(local_save_path)
tokenizer.save_pretrained(local_save_path)
print("✅ تم الحفظ المحلي بنجاح.")

# 3. الرفع باستخدام HfApi
api = HfApi(token=hf_token)
# 🔄 إرجاع اسم المستودع إلى الاسم الأصلي
repo_name = "ameer4wisam/gemma-iraqi-finetune"

try:
    # التأكد من وجود المستودع
    api.create_repo(repo_id=repo_name, private=False, exist_ok=True)
    print(f"✅ تم التأكد من وجود المستودع: {repo_name}")

    print("جاري رفع المجلد إلى Hugging Face لتحديث الإصدار (قد يستغرق بعض الوقت)...")
    api.upload_folder(
        folder_path=local_save_path,
        repo_id=repo_name,
        repo_type="model"
    )
    print(f"\n🎉 تم تحديث النموذج بنجاح! الرابط:\nhttps://huggingface.co/{repo_name}")
except Exception as e:
    print(f"\n⚠️ حدث خطأ أثناء الرفع:")
    print(e)

In [ ]:
messages = [{"role": "user", "content": "شلونك شخبارك"}]
outputs = pipe(messages, max_new_tokens=256, disable_compile=True)
print(outputs[0]["generated_text"][-1]["content"])

In [ ]:
for i in range(10):
    out = pipe([{"role": "user", "content": "شلونك شخبارك"}],
               max_new_tokens=64, disable_compile=True)
    print(f"{i+1}. {out[0]['generated_text'][-1]['content']}")

In [ ]:
# ============================================================
#  خلية التوليد المثالية - Gemma Iraqi Dialect Testing
# ============================================================
import torch

# ---------- 1) إعدادات التوليد ----------
# وضعين: حتمي (للتشخيص) و إبداعي (للاستخدام الفعلي)

GEN_CONFIG_DETERMINISTIC = dict(
    max_new_tokens=512,
    do_sample=False,              # حتمي 100% - نفس المدخل يعطي نفس المخرج
    repetition_penalty=1.1,       # يمنع التكرار الممل
)

GEN_CONFIG_BALANCED = dict(
    max_new_tokens=512,
    do_sample=True,
    temperature=0.7,              # توازن: طبيعي بدون هلوسة
    top_p=0.9,                    # nucleus sampling - يقص الذيل الغريب
    top_k=50,                     # حد أقصى للمرشحين
    repetition_penalty=1.1,
)

# نقطة التوقف الصحيحة لـ Gemma
EOT_ID = tokenizer.convert_tokens_to_ids("<end_of_turn>")

# ---------- 2) دالة التوليد مع السياق الكامل ----------
def chat(messages, mode="balanced", show_prompt=False):
    """
    messages: قائمة بصيغة [{"role": "user", "content": "..."}, ...]
    mode: "deterministic" للتشخيص | "balanced" للاستخدام الطبيعي
    """
    config = GEN_CONFIG_DETERMINISTIC if mode == "deterministic" else GEN_CONFIG_BALANCED

    # نطلب قاموس صراحةً ونستخرج منه input_ids و attention_mask
    encoded = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,             # <-- التعديل الأساسي
    )
    input_ids = encoded["input_ids"].to(model.device)
    attention_mask = encoded["attention_mask"].to(model.device)

    if show_prompt:
        print("=== البرومبت الفعلي الداخل للموديل ===")
        print(tokenizer.decode(input_ids[0], skip_special_tokens=False))
        print("=" * 50)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,   # يمنع تحذيرات الـ padding
            eos_token_id=EOT_ID,
            pad_token_id=tokenizer.pad_token_id or EOT_ID,
            **config,
        )

    # نقص المدخل ونفك ترميز الجواب الجديد فقط
    reply = tokenizer.decode(
        outputs[0][input_ids.shape[1]:],
        skip_special_tokens=True,
    ).strip()
    return reply
# ---------- 3) دالة محادثة تفاعلية متعددة الأدوار ----------
def run_conversation(user_turns, mode="balanced", verbose=True):
    """
    تدير محادثة كاملة: كل رد يدخل بالسياق للدور اللي بعده
    """
    messages = []
    for i, user_msg in enumerate(user_turns, 1):
        messages.append({"role": "user", "content": user_msg})
        reply = chat(messages, mode=mode)
        messages.append({"role": "assistant", "content": reply})
        if verbose:
            print(f"👤 المستخدم [{i}]: {user_msg}")
            print(f"🤖 الموديل  [{i}]: {reply}")
            print("-" * 60)
    return messages


# ============================================================
#  4) الاختبارات
# ============================================================

# ---------- اختبار أ: محادثة مبيعات طويلة (نقطة القوة) ----------
print("\n" + "🔵 اختبار 1: محادثة مبيعات طويلة (وضع متوازن)".center(60, "=") + "\n")
sales_conversation = [
    "هلو، شلونكم؟ عندكم مكيفات سبلت؟",
    "زين شكد سعر الطن ونص؟",
    "غالي شوية.. ماكو أرخص منه؟",
    "والفرق بالجودة بين الاثنين شنو؟",
    "طيب إذا أخذت اثنين، تعطيني خصم؟",
    "زين والتركيب عليكم لو علي؟",
    "تمام خلي أفكر وأرد عليكم، شكراً",
]
run_conversation(sales_conversation, mode="balanced")

# ---------- اختبار ب: نفس المحادثة بالوضع الحتمي (للمقارنة) ----------
print("\n" + "🟢 اختبار 2: أسئلة متابعة بالوضع الحتمي".center(60, "=") + "\n")
followup_test = [
    "عندكم غسالات اتوماتيك؟",
    "وتعطي خصم للاثنين؟",        # سؤال المتابعة اللي فشل سابقاً - هسه بسياق
]
run_conversation(followup_test, mode="deterministic")

# ---------- اختبار ج: التحيات (الأمثلة المضافة بـ v4) ----------
print("\n" + "🟡 اختبار 3: التحيات والمجاملات".center(60, "=") + "\n")
greetings_test = [
    "هلو شلونك؟",
    "الحمدلله، اشتريت البضاعة وعجبتني هواي",
    "الله يحفظك، فمان الله",
]
run_conversation(greetings_test, mode="balanced")

# ---------- اختبار د: القدرة العامة (نقطة الضعف المعروفة) ----------
print("\n" + "🔴 اختبار 4: أسئلة عامة خارج نطاق المبيعات".center(60, "=") + "\n")
general_test = [
    "شنو عاصمة فرنسا؟",
    "اشرحلي بالعراقي شنو يعني ذكاء اصطناعي بجملتين",
]
run_conversation(general_test, mode="deterministic")

print("\n✅ كل الاختبارات خلصت")

In [ ]:
# ============================================================
#  كود الاستدلال الصحيح - Gemma 4 Iraqi Dialect
# ============================================================
import torch

# ---------- 1) اكتشاف توكن التوقف الصحيح تلقائياً ----------
# نطبق القالب على محادثة صغيرة، وآخر توكنات النص هي علامة نهاية الدور الحقيقية
_probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "هلو"},
     {"role": "assistant", "content": "هلا بيك"}],
    add_generation_prompt=False,
    return_tensors="pt" # Ensure it returns a tensor, which will be a BatchEncoding object
)

# نطبع آخر التوكنات للفحص البصري
print("آخر 5 توكنات بعد رد الموديل:")
for tid in _probe["input_ids"][0][-5:]: # Access input_ids and then slice the tensor
    print(f"  id={tid}  ->  {repr(tokenizer.decode([tid]))}")

# نجمع كل التوكنات الخاصة الموجودة بنهاية القالب كمرشحين للتوقف
special_ids = set(tokenizer.all_special_ids)
stop_ids = [tid for tid in _probe["input_ids"][0][-3:] if tid in special_ids or "turn" in tokenizer.decode([tid])] # Access input_ids for stop_ids
if tokenizer.eos_token_id is not None:
    stop_ids.append(tokenizer.eos_token_id)
stop_ids = list(set(stop_ids))

print(f"\n✅ توكنات التوقف المعتمدة: {stop_ids}")
print(f"   -> {[repr(tokenizer.decode([t])) for t in stop_ids]}")

PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else stop_ids[0]


# ---------- 2) إعدادات التوليد ----------
# max_new_tokens=64 لأن أجوبة بياناتك قصيرة (10-30 توكن غالباً)
# بدون repetition_penalty نهائياً - يخرب مفردات اللهجة

GEN_DETERMINISTIC = dict(
    max_new_tokens=64,
    do_sample=False,
)
GEN_LIGHT = dict(
    max_new_tokens=64,
    do_sample=True,
    temperature=0.45,
    top_p=0.85,
    top_k=30,
)
GEN_BALANCED = dict(
    max_new_tokens=64,
    do_sample=True,
    temperature=0.3,      # كان 0.6 - ننزل النص
    top_p=0.8,            # كان 0.9 - نقص الذيل أكثر
    top_k=20,             # نحصر المرشحين بأفضل 20 توكن بس
)

# ---------- 3) دالة التوليد ----------
def chat(messages, mode="balanced", show_prompt=False):
    config = GEN_DETERMINISTIC if mode == "deterministic" else GEN_BALANCED

    enc = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    input_ids = enc["input_ids"].to(model.device)
    attention_mask = enc["attention_mask"].to(model.device)

    if show_prompt:
        print("=== البرومبت الداخل للموديل ===")
        print(tokenizer.decode(input_ids[0], skip_special_tokens=False))
        print("=" * 50)

    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            eos_token_id=stop_ids,      # <-- التوقف الصحيح
            pad_token_id=PAD_ID,
            **config,
        )

    reply = tokenizer.decode(
        out[0][input_ids.shape[1]:],
        skip_special_tokens=True,
    ).strip()
    return reply


# ---------- 4) محادثة متعددة الأدوار ----------
def run_conversation(user_turns, mode="balanced"):
    messages = []
    for i, user_msg in enumerate(user_turns, 1):
        messages.append({"role": "user", "content": user_msg})
        reply = chat(messages, mode=mode)
        messages.append({"role": "assistant", "content": reply})
        print(f"👤 [{i}]: {user_msg}")
        print(f"🤖 [{i}]: {reply}")
        print("-" * 60)
    return messages


# ============================================================
#  5) الاختبارات
# ============================================================

# اختبار أولي: سؤال واحد بالوضع الحتمي (أهم اختبار)
print("\n" + "🟢 اختبار حتمي - سؤال واحد".center(60, "=") + "\n")
print("🤖:", chat([{"role": "user", "content": "هلو، عندكم مكيفات سبلت؟"}], mode="deterministic"))

# محادثة متابعة بالوضع الحتمي
print("\n" + "🟢 اختبار حتمي - متابعة بسياق".center(60, "=") + "\n")
run_conversation([
    "عندكم غسالات اتوماتيك؟",
    "وتعطي خصم للاثنين?",
], mode="deterministic")

# محادثة مبيعات بالوضع المتوازن
print("\n" + "🔵 اختبار متوازن - محادثة مبيعات".center(60, "=") + "\n")
run_conversation([
    "هلو، عندكم ثلاجات؟",
    "شكد سعر أرخص وحدة؟",
    "غالي شوية، ماكو تنزيل؟",
], mode="balanced")

# تحيات
print("\n" + "🟡 اختبار التحيات".center(60, "=") + "\n")
run_conversation([
    "هلو شلونك؟",
    "الله يحفظك، فمان الله",
], mode="balanced")

In [ ]:
# ============================================================
#  اختبار السياق الطويل - نفس صيغة الاستدلال
# ============================================================

# دالة معدلة توري طول السياق بكل دور
def run_long_conversation(user_turns, mode="deterministic"):
    messages = []
    for i, user_msg in enumerate(user_turns, 1):
        messages.append({"role": "user", "content": user_msg})

        # نحسب طول البرومبت بالتوكنات قبل التوليد
        probe = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True
        )
        ctx_len = len(probe)

        reply = chat(messages, mode=mode)
        messages.append({"role": "assistant", "content": reply})

        # علامة تحذير إذا تجاوزنا أطول مثال بالتدريب (212 توكن)
        flag = "⚠️ خارج نطاق التدريب" if ctx_len > 212 else "✅"
        print(f"👤 [{i}] (سياق: {ctx_len} توكن {flag}): {user_msg}")
        print(f"🤖 [{i}]: {reply}")
        print("-" * 60)
    return messages


# ---------- اختبار 1: محادثة مبيعات طويلة 12 دور (حتمي) ----------
print("\n" + "🟢 اختبار 1: مبيعات 12 دور - حتمي".center(60, "=") + "\n")
long_sales = [
    "هلو، شلونكم؟",
    "عندكم مكيفات سبلت؟",
    "شنو الماركات الموجودة؟",
    "شكد سعر الطن الواحد؟",
    "والطن ونص شكد؟",
    "الفرق بالكهرباء بينهم هواي؟",
    "زين، والضمان شكد؟",
    "إذا أخذت اثنين تنزلي بالسعر؟",
    "والتركيب عليكم لو علي؟",
    "التوصيل لأي منطقة؟",
    "زين شوكت أكدر أستلمهم؟",
    "تمام، اتفقنا. فمان الله",
]
run_long_conversation(long_sales, mode="deterministic")

# ---------- اختبار 2: نفس المحادثة بالوضع المتوازن الضيق ----------
print("\n" + "🔵 اختبار 2: نفس المحادثة - متوازن (0.3)".center(60, "=") + "\n")
run_long_conversation(long_sales, mode="balanced")

# ---------- اختبار 3: اختبار ذاكرة السياق ----------
# نذكر معلومة بأول المحادثة ونسأل عنها بعد عدة أدوار
print("\n" + "🟣 اختبار 3: ذاكرة السياق - حتمي".center(60, "=") + "\n")
memory_test = [
    "هلو، اسمي أبو علي وأدور على ثلاجة",
    "شنو الأحجام الموجودة؟",
    "شكد سعر الكبيرة؟",
    "زين والصغيرة؟",
    "والألوان شنو؟",
    "طيب، تتذكر شنو اسمي وشنو كنت أدور؟",   # <-- سؤال الذاكرة
]
run_long_conversation(memory_test, mode="deterministic")

In [ ]:
# ============================================================
#  اختبار السياق الطويل + System Prompt
# ============================================================

IRAQI_SYSTEM_PROMPT = """أنت بائع عراقي محترف بمحل أجهزة كهربائية.

قواعد صارمة قبل كل رد:
1. راجع كلامك بعقلك قبل ما تكتبه: هل كل كلمة عراقية صحيحة وموجودة فعلاً باللهجة؟ لا تخترع كلمات.
2. استخدم مفردات عراقية أصيلة: شنو، شلون، هسه، اكو، ماكو، زين، خوش، هواي، شكد، چا، هيچي.
3. جاوب قصير ومباشر مثل البائع الحقيقي، جملة أو جملتين بس.
4. إذا ذكرت سعر أو رقم بالمحادثة، التزم بيه ولا تغيره بالأدوار الجاية.
5. تذكر تفاصيل الزبون (اسمه، شنو يريد) واستعملها بردودك.
6. إذا ما تعرف معلومة، گول "أتأكدلك" ولا تخترع جواب.
7. بِع فقط المنتجات المذكورة بالقائمة أعلاه. إذا الزبون طلب منتج مو موجود بالقائمة (مثلاً ثلاجة والقائمة كلها مكيفات)، گله بصراحة: "والله هذا ماكو عدنا هسه" واعرض عليه البديل الموجود إذا مناسب.

"""


# دالة معدلة: عداد توكنات مصلّح + دعم system prompt
def run_long_conversation(user_turns, mode="deterministic", use_system=True):
    messages = []
    if use_system:
        messages.append({"role": "system", "content": IRAQI_SYSTEM_PROMPT})

    for i, user_msg in enumerate(user_turns, 1):
        messages.append({"role": "user", "content": user_msg})

        # ✅ تصحيح العداد: return_dict ونقرأ input_ids
        probe = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_dict=True
        )
        ctx_len = len(probe["input_ids"])

        reply = chat(messages, mode=mode)
        messages.append({"role": "assistant", "content": reply})

        flag = "⚠️ خارج نطاق التدريب" if ctx_len > 212 else "✅"
        print(f"👤 [{i}] (سياق: {ctx_len} توكن {flag}): {user_msg}")
        print(f"🤖 [{i}]: {reply}")
        print("-" * 60)
    return messages


long_sales = [
    "هلو، شلونكم؟",
    "عندكم مكيفات سبلت؟",
    "شنو الماركات الموجودة؟",
    "شكد سعر الطن الواحد؟",
    "والطن ونص شكد؟",
    "الفرق بالكهرباء بينهم هواي؟",
    "زين، والضمان شكد؟",
    "إذا أخذت اثنين تنزلي بالسعر؟",
    "والتركيب عليكم لو علي؟",
    "التوصيل لأي منطقة؟",
    "زين شوكت أكدر أستلمهم؟",
    "تمام، اتفقنا. فمان الله",
]

memory_test = [
    "هلو، اسمي أبو علي وأدور على ثلاجة",
    "شنو الأحجام الموجودة؟",
    "شكد سعر الكبيرة؟",
    "زين والصغيرة؟",
    "والألوان شنو؟",
    "طيب، تتذكر شنو اسمي وشنو كنت أدور؟",
]

# ---------- اختبار 1: مبيعات 12 دور - حتمي + system ----------
print("\n" + "🟢 اختبار 1: مبيعات 12 دور - حتمي + system".center(60, "=") + "\n")
run_long_conversation(long_sales, mode="deterministic", use_system=True)

# ---------- اختبار 2: نفس المحادثة - متوازن + system ----------
print("\n" + "🔵 اختبار 2: متوازن (0.3) + system".center(60, "=") + "\n")
run_long_conversation(long_sales, mode="balanced", use_system=True)

# ---------- اختبار 3: الذاكرة - حتمي + system ----------
# القاعدة 5 بالبرومبت تستهدف بالضبط فشل "أبو علي" السابق
print("\n" + "🟣 اختبار 3: ذاكرة السياق + system".center(60, "=") + "\n")
run_long_conversation(memory_test, mode="deterministic", use_system=True)

# ---------- اختبار 4 (مرجعي): الذاكرة بدون system للمقارنة ----------
print("\n" + "⚪ اختبار 4: الذاكرة بدون system (مرجع)".center(60, "=") + "\n")
run_long_conversation(memory_test, mode="deterministic", use_system=False)

In [ ]:
import os
import matplotlib.pyplot as plt
from huggingface_hub import HfApi
from google.colab import userdata

# ============================================================
# 1. إعادة رسم منحنى الخسارة وحفظه كصورة للرفع
# ============================================================
if 'trainer' in globals():
    log_history = trainer.state.log_history
    train_losses = [log["loss"] for log in log_history if "loss" in log]
    epoch_train = [log["epoch"] for log in log_history if "loss" in log]
    eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]
    epoch_eval = [log["epoch"] for log in log_history if "eval_loss" in log]

    plt.figure(figsize=(10, 6))
    plt.plot(epoch_train, train_losses, label="Training Loss")
    plt.plot(epoch_eval, eval_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss per Epoch")
    plt.legend()
    plt.grid(True)
    plt.savefig("loss_curve.png")
    plt.close()
    print("✅ تم حفظ صورة منحنى الخسارة محلياً.")
else:
    print("⚠️ لم يتم العثور على المدرب (trainer)، تأكد من تشغيل خلية التدريب.")

# ============================================================
# 2. إنشاء محتوى Model Card الشامل
# ============================================================
repo_name = "ameer4wisam/gemma-iraqi-finetune"

model_card_content = f"""---
language:
- ar
- en
license: gemma
base_model: google/gemma-4-E4B-it
tags:
- iraqi
- arabic
- dialect
- lora
- peft
---

# نموذج جيما للهجة العراقية (Gemma 4B - Iraqi Dialect)

هذا النموذج هو نسخة محسّنة (Fine-tuned) من نموذج `google/gemma-4-E4B-it`، تم تدريبه خصيصاً لفهم والتحدث بـ **اللهجة العراقية** باستخدام تقنية **LoRA** (Low-Rank Adaptation).

## 📌 تفاصيل النموذج والتدريب
* **النموذج الأساسي:** [google/gemma-4-E4B-it](https://huggingface.co/google/gemma-4-E4B-it)
* **عدد الحقب (Epochs):** 1
* **معدل التعلم (Learning Rate):** 5e-5 مع مجدول `cosine`
* **حجم الدفعة (Batch Size):** 8
* **الـ LoRA Rank (r):** 16 | **Alpha:** 32
* **إعدادات خاصة:** تفعيل `use_cache=True` لمعمارية E4B.

### منحنى الخسارة (Loss Curve)
![Loss Curve](loss_curve.png)

## 🚀 كيفية الاستخدام (الاستدلال المفرد وضمن السياق الطويل)

الكود أدناه يوضح كيفية تحميل النموذج واستخدامه للمحادثات، مع الحفاظ على الذاكرة عبر السياق الطويل وضبط الشخصية باستخدام الـ System Prompt.

```python
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from peft import PeftModel

# 1. تحميل النموذج الأساسي والمحوّل (Adapter)
base_model_id = "google/gemma-4-E4B-it"
adapter_model_id = "{repo_name}"

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = PeftModel.from_pretrained(base_model, adapter_model_id)

# 2. تحديد ثوابت التوقف (لمنع الهلوسة)
stop_ids = [106, 1]  # <turn|> و <eos>
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else stop_ids[0]

# 3. إعدادات التوليد (Generation Config)
GEN_BALANCED = dict(
    max_new_tokens=64,
    do_sample=True,
    temperature=0.3,
    top_p=0.8,
    top_k=20,
)

# 4. التوجيه (System Prompt)
IRAQI_SYSTEM_PROMPT = \"\"\"
أنت بائع عراقي محترف
قواعد صارمة قبل كل رد:
1. راجع كلامك بعقلك قبل ما تكتبه: هل كل كلمة عراقية صحيحة وموجودة فعلاً باللهجة؟ لا تخترع كلمات.
2. استخدم مفردات عراقية أصيلة: شنو، شلون، هسه، اكو، ماكو، زين، خوش، هواي، شكد، چا، هيچي.
3. جاوب قصير ومباشر مثل البائع الحقيقي، جملة أو جملتين بس.
4. إذا ذكرت سعر أو رقم بالمحادثة، التزم بيه ولا تغيره بالأدوار الجاية.
5. تذكر تفاصيل الزبون (اسمه، شنو يريد) واستعملها بردودك.
6. إذا ما تعرف معلومة، گول "أتأكدلك" ولا تخترع جواب.
7. بِع فقط المنتجات المذكورة بالقائمة أعلاه. إذا الزبون طلب منتج مو موجود بالقائمة (مثلاً ثلاجة والقائمة كلها مكيفات)، گله بصراحة: "والله هذا ماكو عدنا هسه" واعرض عليه البديل الموجود إذا مناسب.

\"\"\"

# 5. دالة الاستدلال الأساسية
def chat(messages):
    enc = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    input_ids = enc["input_ids"].to(model.device)
    attention_mask = enc["attention_mask"].to(model.device)

    with torch.no_grad():
        out = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            eos_token_id=stop_ids,
            pad_token_id=PAD_ID,
            **GEN_BALANCED,
        )

    reply = tokenizer.decode(
        out[0][input_ids.shape[1]:],
        skip_special_tokens=True,
    ).strip()
    return reply

# 6. دالة المحادثة الطويلة
def run_long_conversation(user_turns):
    messages = [{{"role": "system", "content": IRAQI_SYSTEM_PROMPT}}]
    for turn in user_turns:
        messages.append({{"role": "user", "content": turn}})
        reply = chat(messages)
        messages.append({{"role": "assistant", "content": reply}})
        print(f"👤: {{turn}}\n🤖: {{reply}}\n--- ")

# === تجربة الاستدلال ===
run_long_conversation([
    "هلو، اسمي أبو علي وأدور على ثلاجة",
    "شنو الأحجام الموجودة؟",
    "طيب، تتذكر شنو اسمي وشنو كنت أدور؟"
])
```
"""

# حفظ النص في ملف README.md محلياً
with open("README.md", "w", encoding="utf-8") as f:
    f.write(model_card_content)
print("✅ تم إنشاء التوثيق الشامل (README.md).")

# ============================================================
# 3. رفع التوثيق والصورة إلى Hugging Face
# ============================================================
try:
    hf_token = userdata.get('ameerwisam2005')
    api = HfApi(token=hf_token)

    print("\nجاري رفع صورة منحنى الخسارة والتوثيق الجديد إلى Hugging Face...")

    # رفع صورة المنحنى إذا كانت موجودة
    if os.path.exists("loss_curve.png"):
        api.upload_file(
            path_or_fileobj="loss_curve.png",
            path_in_repo="loss_curve.png",
            repo_id=repo_name,
            repo_type="model",
        )
        print("✅ تم رفع صورة منحنى الخسارة.")

    # رفع التوثيق
    api.upload_file(
        path_or_fileobj="README.md",
        path_in_repo="README.md",
        repo_id=repo_name,
        repo_type="model",
    )
    print(f"✅ تم تحديث التوثيق بنجاح! تفقد الرابط: https://huggingface.co/{repo_name}")
except Exception as e:
    print(f"حدث خطأ أثناء الرفع: {e}")
